[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/octonions/octonions.ipynb)

# Octonions and Their Symmetries

This example builds the octonions, which with the real numbers, the complex numbers and the quaternions are the only number systems in which the length of a product is the product of the lengths, so that any number other than zero can be divided by. They have eight components, and each step up the line gives something away: the complex numbers cannot be put in order, the quaternions do not commute, and the octonions are not associative either, so that in a product of three the brackets matter. The rotations of their seven imaginary directions that keep the product intact form the exceptional group G₂, fourteen of the twenty-one dimensions of rotations.

Every geometric algebra is associative, so the octonions are not one, but they are built from one. An octonion is modelled as a scalar plus a vector of the algebra of seven directions that square to minus one, and the seven multiplication triples as a single trivector, `orientation`. The product is then one extensor, `multiply`, a map from pairs of octonions to octonions. Multiplication by a single octonion, `multiply(mv.a, Octonion)`, is a map on octonions, and maps compose associatively even where the product does not. The symmetries are the bivectors whose commutator with `orientation` vanishes, the null space of the map `Bivector.commutator(orientation)`. The same octonions also fit whole into the eight components of the algebra of three-dimensional space.

In index notation the imaginary units multiply as $e_i e_j = -\delta_{ij} + \varepsilon_{ijk} e_k$, with $\varepsilon_{ijk}$ totally antisymmetric and equal to one on each of the seven triples, and $G_2$ is the group of rotations of $\mathbb R^7$ that leave $\varepsilon_{ijk}$ unchanged.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import VGA3D as space
from examples.animation import save_figure
from examples.math.octonions import render

In [ ]:
ga = Algebra("a-b-c-d-e-f-g-")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Octonion = ga.gatype.from_blades("1 a b c d e f g")
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
# An octonion as a whole multivector of space, Cl(3, 0).
SpaceOctonion = space.gatype.from_blades("1 x y z yz zx xy xyz")

## 1. The octonion product

The octonion components are the scalar and the seven vectors. Seven oriented triples give the multiplication: `a` times `b` is `c`, `b` times `c` is `a`, and reversing the order changes the sign. Summed, the triples are one trivector, which carries the whole multiplication table; the product is the geometric product with one minus that trivector, keeping the scalar and vector parts. That projection belongs to the octonion product, and the geometric product itself stays associative.

Drawn with a point for each basis vector and a line for each triple, the seven triples make the Fano plane, the smallest arrangement of points and lines in which every two points lie on one line and every two lines meet in one point; each arrow runs in the order of the product.

In [ ]:
# Each triple supplies one quaternion cycle; every pair of basis vectors occurs once.
cycles = stack([mv.abc, mv.ade, mv.agf, mv.bdf, mv.beg, mv.cdg, mv.cfe])         # [triples] Trivector
orientation = cycles.sum(axis=0)                                              # [] Trivector
# The geometric product of two octonions with one minus the trivector, cut back to the octonion components.
multiply = ((Octonion * Octonion) * (1 - orientation)).cast(Octonion)          # [] Octonion <- (Octonion, Octonion)

print("a times b")
print(multiply(mv.a, mv.b).coefficient_table())
print("b times a")
print(multiply(mv.b, mv.a).coefficient_table())
save_figure(render.draw_fano(cycles), "octonions");

## 2. Lengths and brackets

The squared length of an octonion is its scalar product with its grade involution, which negates the seven vectors and so makes up for their negative squares. The length of a product is the product of the lengths, even though the two bracketings of three factors can disagree. Any two octonions still generate an associative subalgebra; the brackets matter only for three independent factors.

In [ ]:
# Two octonions.
first = 1 + 2 * mv.a - mv.d
second = mv.b + mv.f

In [ ]:
def norm_squared(value: Octonion) -> Scalar:
    """Positive squared length, pairing the value with its grade involution."""
    return value.involute().scalar_product(value)


def associator(first: Octonion, second: Octonion, third: Octonion) -> Octonion:
    """The difference between the two bracketings of three factors."""
    return multiply(multiply(first, second), third) - multiply(first, multiply(second, third))


print("Squared length of the product")
print(norm_squared(multiply(first, second)).coefficient_table())
print("Product of the squared lengths")
print((norm_squared(first) * norm_squared(second)).coefficient_table())
print("(a b) d")
print(multiply(multiply(mv.a, mv.b), mv.d).coefficient_table())
print("a (b d)")
print(multiply(mv.a, multiply(mv.b, mv.d)).coefficient_table())

## 3. Multiplication by an octonion as a map

Binding the second factor to `Octonion`, the space of all octonions, makes multiplication by `a` a map on octonions, `multiply(mv.a, Octonion)`. Applied twice it is minus the identity, and the maps of two different basis vectors anticommute, so the seven maps form an extensor representation of Cl(0, 7) on the eight octonion components. Composing maps is associative, so composing multiplication by `a` and by `b` need not be multiplication by their octonion product.

In [ ]:
left_a = multiply(mv.a, Octonion)                                   # [] Octonion <- Octonion
left_b = multiply(mv.b, Octonion)                                   # [] Octonion <- Octonion
# Composition keeps its brackets: this applies the inner multiplication first.
square = left_a(left_a)                                             # [] Octonion <- Octonion
anticommutator = left_a(left_b) + left_b(left_a)                    # [] Octonion <- Octonion

print("Left multiplication by a")
print(left_a.coefficient_table())
print("Its square")
print(square.coefficient_table())
print("Composed with multiplication by b, both ways round, and summed")
print(anticommutator.coefficient_table())

## 4. Rotations that preserve multiplication

A transformation that preserves multiplication fixes the scalar one and preserves length, so it is a rotation of the seven vectors. Infinitesimally it obeys the product rule: acting on a product equals acting on each factor in turn and adding the results. Rotations already commute with the geometric product and with the selection of grades, so only the fixed trivector can break the rule, and the symmetries are the bivectors that leave it unchanged. Their commutator with it, `Bivector.commutator(orientation)`, is a map from bivectors to trivectors, and its null space holds the symmetries.

In [ ]:
# The singular values counted as zero, the plane of an ordinary rotation, and the angle of the finite turns.
tolerance = 1e-10
rotation_plane = mv.ab
angle = 0.8

In [ ]:
constraint = Bivector.commutator(orientation)                       # [] Trivector <- Bivector
_, singular_values, directions = constraint.svd()
generators = directions[singular_values < tolerance]                 # [generators] Bivector
print(f"{len(generators)} of {len(directions)} rotation generators preserve multiplication.")

# Project the ordinary rotation onto that space, summing its components over the batch.
symmetry = (generators * generators.reverse().scalar_product(rotation_plane)).sum(axis=0)  # Bivector
derivation = symmetry.commutator(Octonion)                          # [] Vector <- Octonion
# Acting on a product equals acting on each factor in turn: this binary map vanishes.
product_rule = (derivation(multiply) - multiply(derivation, Octonion)
                - multiply(Octonion, derivation))
with np.printoptions(precision=3):
    print("An infinitesimal symmetry")
    print(derivation.coefficient_table())

The fourteen directions form the Lie algebra of the exceptional group $G_2$. Their commutators stay in the same space. Exponentiating one gives a finite rotation that preserves octonion multiplication; an arbitrary rotation usually does not.

In [ ]:
rotations = (stack([rotation_plane, symmetry]) * (angle / 2)).exp()            # [cases] Rotor
# Compare moving the product with multiplying the moved factors.
rotated_product = rotations >> multiply(mv.a, mv.d)
product_of_rotated = multiply(rotations >> mv.a, rotations >> mv.d)
difference = rotated_product - product_of_rotated
with np.printoptions(precision=3):
    print("Ordinary rotation: difference")
    print(difference[0].coefficient_table())
    print("G₂ rotation: difference")
    print(difference[1].coefficient_table())

## 5. The same octonions in the algebra of space

The eight components of the geometric algebra of three-dimensional space hold an octonion whole, by doubling the quaternions. The even part of a multivector of space, a scalar plus a bivector, is modelled as a quaternion, and the odd part, a vector plus a trivector, as the pseudoscalar times a second quaternion. The product multiplies the even parts as quaternions and brings in the odd parts reversed and in swapped order, and the squared length is `value.scalar_norm_squared()`. Choosing x, y and z for `a`, `b` and `d` fixes the other four units through the triples, and all seven triples then hold. With z squaring to minus one instead, the same product gives the split octonions, in which a number other than zero can have length zero and multiply its own conjugate, `scalar_negation()`, to zero.

In the notation of Cayley and Dickson an octonion is a pair of quaternions, $(p, q)(r, s) = (pr - \bar s q,\ sp + q \bar r)$; the second of each pair is the odd part here.

In [ ]:
space_mv = NumpyContext(space).multivector


def space_multiply(first: SpaceOctonion, second: SpaceOctonion) -> SpaceOctonion:
    """The octonion product of two multivectors of space, by doubling the quaternions."""
    # The even parts are quaternions; the odd parts, the pseudoscalar times a second quaternion apiece.
    first_even, first_odd = first.restrict.even(), first.restrict.odd()
    second_even, second_odd = second.restrict.even(), second.restrict.odd()
    return (first_even * second_even - second_odd.reverse() * first_odd
            + second_odd * first_even + first_odd * second_even.reverse())


# x, y and z stand for a, b and d; the triples abc, ade, bdf and cdg make c, e, f and g their products.
a, b, d = space_mv.x, space_mv.y, space_mv.z
c, e, f = space_multiply(a, b), space_multiply(a, d), space_multiply(b, d)
g = space_multiply(c, d)
images = stack([a, b, c, d, e, f, g])                                         # [images] SpaceOctonion
# The other three triples, agf, beg and cfe, then hold by themselves.
print("a g - f:", render.largest(space_multiply(a, g) - f), " b e - g:", render.largest(space_multiply(b, e) - g),
      " c f - e:", render.largest(space_multiply(c, f) - e))

split = Algebra("x+y+z-")
split_mv = NumpyContext(split).multivector
null = 1 + split_mv.z                                                          # [] SpaceOctonion, split
print("In the split octonions, the length of 1 + z:", render.scalars(null.scalar_norm_squared()))
print("and 1 + z times its conjugate:", render.largest(space_multiply(null, null.scalar_negation())))

The Clifford construction and the seven oriented triples follow Lemma 2.5 of [Kauhanen and Orelma, *On the Structure of Octonion Regular Functions*](https://doi.org/10.1007/s00006-019-0983-y). John Baez explains the [Clifford actions](https://math.ucr.edu/home/baez/octonions/node6.html) and the [octonion symmetries and derivations](https://math.ucr.edu/home/baez/octonions/node14.html) in *The Octonions*.

The construction in the algebra of space follows Daniele Corradetti, *Recovering Composition Algebras from 3D Geometric Algebras* (2024), who derived it from Anthony Lasenby's octonions in the even subalgebra of the spacetime algebra, *Some recent results for SU(3) and Octonions within the Geometric Algebra approach to the fundamental forces of nature* (2022); [this answer](https://math.stackexchange.com/a/5037113) lays out how the two meet.

In [ ]:
# checks
# Lengths multiply, and the two bracketings of a, b and d differ by twice g.
assert abs(render.scalars(norm_squared(multiply(first, second)) - norm_squared(first) * norm_squared(second))) < 1e-10
assert render.largest(associator(mv.a, mv.b, mv.d) - 2 * mv.g) < 1e-10
# Multiplication by a squares to minus the identity, and multiplications by a and by b anticommute.
assert render.largest(square + Octonion) < 1e-10 and render.largest(anticommutator) < 1e-10
# Fourteen generators, the symmetry among them obeying the product rule; its rotation keeps the product, the
# ordinary rotation does not.
assert len(generators) == 14 and render.largest(product_rule) < 1e-10
assert render.largest(difference[1]) < 1e-10 and render.largest(difference[0]) > 0.1
# In the algebra of space, every image squares to minus one, every two anticommute, and the three triples not
# used to build them hold.
both_orders = space_multiply(images[:, None], images[None, :]) + space_multiply(images[None, :], images[:, None])   # [images, images] SpaceOctonion
apart = ~np.eye(images.shape[0], dtype=bool)
assert render.largest(space_multiply(images, images) + 1) < 1e-10 and render.largest(both_orders[apart]) < 1e-10
assert max(render.largest(space_multiply(a, g) - f), render.largest(space_multiply(b, e) - g), render.largest(space_multiply(c, f) - e)) < 1e-10
# In the split octonions, 1 + z has length zero and multiplies its conjugate to zero.
assert render.largest(null.scalar_norm_squared()) < 1e-10 and render.largest(space_multiply(null, null.scalar_negation())) < 1e-10